[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C22_Reasoning_RL_Course/04_verifier_search/04_verifier_search.ipynb)

# 04 · Verifier 引导搜索（用 numpy 模拟）

目标：从零实现**测试时搜索**——best-of-N、(加权)多数投票、beam search——并把 **pass@k 的无偏估计**做对(对拍组合数闭式解、揭示 $1-(1-\hat p)^k$ 的偏差)。

路线：BoN(对拍上界) → 多数投票 → 加权投票(胜过裸 BoN) → pass@k 无偏估计 vs 有偏 → pass@k vs pass^k → beam search → ✏️ 练习 → 📖 答案 → 🧪 真实 pass@k 胶囊。

> 心智模型：**搜索 = 生成多个候选 + verifier 挑/引导**，前提是『判别比生成易』。pass@k 是上界(不可部署), 投票/BoN 才是能用的。

## 1 · best-of-N 与它的上界 pass@N

BoN：采 N 个候选、verifier 选最高分。**性能上界 = pass@N**(完美 verifier)；实际看 verifier 多准。
我们模拟带噪声的 verifier，验证 BoN 介于 pass@1(瞎选)与 pass@N(完美选)之间。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_pool(p_correct, N, verifier_noise, seed):
    '''生成 N 个候选：correct[i]∈{0,1}~Bernoulli(p); verifier 分 = correct + 噪声。'''
    r = np.random.default_rng(seed)
    correct = (r.random(N) < p_correct).astype(int)
    vscore = correct + r.normal(0, verifier_noise, N)   # 学习的 verifier(不完美)
    return correct, vscore

def best_of_n(correct, vscore):
    '''选 verifier 分最高的候选, 返回它是否真对。'''
    return int(correct[np.argmax(vscore)])

def pass_at_N_oracle(correct):
    '''上界: N 个里至少一个对(完美 oracle 能选出).'''
    return int(correct.sum() > 0)

p_correct, N = 0.3, 16
trials = 3000
bon_acc = np.mean([best_of_n(*make_pool(p_correct, N, 0.5, s)) for s in range(trials)])
oracle_acc = np.mean([pass_at_N_oracle(make_pool(p_correct, N, 0.5, s)[0]) for s in range(trials)])
print(f'单样本正确率 pass@1      = {p_correct:.3f}')
print(f'best-of-{N}(噪声verifier) = {bon_acc:.3f}')
print(f'pass@{N} (完美 oracle 上界)= {oracle_acc:.3f}')
assert p_correct < bon_acc < oracle_acc, 'BoN 应介于 pass@1 与 pass@N 之间'
print('✅ BoN 用 verifier 换准确率, 但达不到 pass@N 上界(verifier 不完美)')

## 2 · verifier 越准，BoN 越接近上界

BoN 的实际性能完全取决于 verifier 质量。我们扫 verifier 噪声，验证：噪声→0 时 BoN→pass@N 上界；噪声很大时 BoN→pass@1(等于瞎选)。

In [ ]:
p_correct, N, trials = 0.3, 16, 3000
oracle = np.mean([pass_at_N_oracle(make_pool(p_correct, N, 0.5, s)[0]) for s in range(trials)])
print(f'pass@1={p_correct:.2f}  pass@{N}(上界)={oracle:.3f}')
print(f"{'verifier噪声':>12}{'BoN 正确率':>12}")
accs = []
noises = [0.01, 0.2, 0.5, 1.0, 3.0, 10.0]
for noise in noises:
    acc = np.mean([best_of_n(*make_pool(p_correct, N, noise, s)) for s in range(trials)])
    accs.append(acc)
    print(f'{noise:>12}{acc:>12.3f}')
# 噪声小 -> 接近上界; 噪声大 -> 单调退化, 趋向 pass@1(瞎选)
assert accs[0] > oracle - 0.05, '近完美 verifier -> 近 pass@N 上界'
assert accs[-1] < accs[0] - 0.3, 'verifier 越差 BoN 越弱(明显退化)'
assert abs(accs[-1] - p_correct) < 0.08, '极噪声 verifier(noise=10) -> 退化到瞎选 pass@1'
assert all(accs[i] >= accs[i+1] - 0.02 for i in range(len(accs)-1)), 'BoN 随噪声单调下降'
print('✅ verifier 质量是 BoN 的天花板: 完美->pass@N, 全噪声->pass@1')

## 3 · 多数投票：免 verifier 的搜索

对有**唯一答案**的题：采 N 个解、对**最终答案**取众数。原理：对的路径殊途同归(集中)、错的各错各的(分散)。
免 verifier！我们模拟『正确答案集中、错误答案分散』并验证投票提升正确率。

In [ ]:
from collections import Counter

def sample_answers(p_correct, N, n_wrong_modes, seed):
    '''采 N 个答案: 以 p_correct 给出正确答案(=0), 否则均匀落在 n_wrong_modes 种错误答案之一。'''
    r = np.random.default_rng(seed)
    ans = []
    for _ in range(N):
        if r.random() < p_correct:
            ans.append(0)                       # 0 代表正确答案
        else:
            ans.append(int(r.integers(1, 1 + n_wrong_modes)))  # 分散的错误答案
    return ans

def majority_vote(answers):
    '''返回众数答案; 是否正确(==0)。'''
    top = Counter(answers).most_common(1)[0][0]
    return int(top == 0)

p_correct, N, trials = 0.4, 16, 3000
# 错误分散在很多种(n_wrong_modes 大)时投票最有效
vote_acc = np.mean([majority_vote(sample_answers(p_correct, N, 8, s)) for s in range(trials)])
print(f'单样本正确率       = {p_correct:.3f}')
print(f'多数投票(N={N})正确率= {vote_acc:.3f}')
assert vote_acc > p_correct, '错误分散时, 多数投票应提升正确率(免 verifier)'
print('✅ 多数投票: 对的集中、错的分散 -> 众数选出正确答案, 无需 verifier')

## 4 · 加权投票常胜过裸 BoN

加权投票：每条解按 verifier 分**加权投票**(而非每条一票), 选总权重最高的答案。
它兼取『投票鲁棒』+『verifier 判别』。构造 verifier 有噪声的场景, 验证加权投票 ≥ 裸 BoN。

In [ ]:
def weighted_majority_solve(answers, vscores):
    '''按 verifier 分给各答案加权, 选总权重最高的答案; 返回是否正确(==0)。
       (注: 练习2 会实现返回『答案标签』的版本 weighted_majority)'''
    weight = {}
    for a, w in zip(answers, vscores):
        weight[a] = weight.get(a, 0.0) + w
    top = max(weight, key=weight.get)
    return int(top == 0)

def make_pool_with_answers(p_correct, N, n_wrong_modes, vnoise, seed):
    r = np.random.default_rng(seed)
    ans, correct = [], []
    for _ in range(N):
        if r.random() < p_correct:
            ans.append(0); correct.append(1)
        else:
            ans.append(int(r.integers(1, 1 + n_wrong_modes))); correct.append(0)
    correct = np.array(correct)
    vscores = np.clip(correct + r.normal(0, vnoise, N), 0.01, None)  # 正权重
    return ans, correct, vscores

p_correct, N, trials = 0.35, 16, 4000
bon, wmaj, maj = [], [], []
for s in range(trials):
    ans, correct, vs = make_pool_with_answers(p_correct, N, 6, 0.7, s)
    bon.append(int(correct[np.argmax(vs)]))           # 裸 BoN: 押单条最高分
    wmaj.append(weighted_majority_solve(ans, vs))     # 加权投票
    maj.append(majority_vote(ans))                    # 裸投票
bon_acc, wmaj_acc, maj_acc = np.mean(bon), np.mean(wmaj), np.mean(maj)
print(f'裸 BoN(押单条最高分) = {bon_acc:.3f}')
print(f'裸 多数投票          = {maj_acc:.3f}')
print(f'加权多数投票         = {wmaj_acc:.3f}')
assert wmaj_acc >= bon_acc - 0.01, '加权投票应不输于裸 BoN(抗单点误判)'
assert wmaj_acc >= maj_acc - 0.01, '加权投票应不输于裸投票(用上 verifier)'
print('✅ 加权投票兼取『投票鲁棒』+『verifier 判别』, verifier 有噪声时常胜出')

## 5 · pass@k 的无偏估计(组合数) vs 有偏的 $1-(1-\hat p)^k$

**核心技术点**。从 n 个样本(c 个对)估计 pass@k:
- 无偏(Chen 2021): $1-\binom{n-c}{k}/\binom{n}{k}$
- 有偏(常见错误): $1-(1-c/n)^k$

我们对拍无偏估计 vs **大量重复实验的真实 pass@k**, 并揭示有偏版的偏差。

In [ ]:
from math import comb

def pass_at_k_unbiased(n, c, k):
    '''Chen 2021 无偏估计: 1 - C(n-c,k)/C(n,k)。'''
    if n - c < k:
        return 1.0                       # 错的不够 k 个 -> 必抽到对的
    return 1.0 - comb(n - c, k) / comb(n, k)

def pass_at_k_biased(n, c, k):
    '''常见错误: 1-(1-p_hat)^k, p_hat=c/n。'''
    return 1.0 - (1 - c / n) ** k

def true_pass_at_k(p, k, trials=200000, seed=0):
    '''真实 pass@k: 模拟无穷样本, 每次抽 k 个看是否至少一个对。'''
    r = np.random.default_rng(seed)
    draws = (r.random((trials, k)) < p)
    return float(np.mean(draws.any(axis=1)))

p, n, k = 0.2, 8, 4              # 难题(p小)、小样本 -> 偏差明显
truth = true_pass_at_k(p, k)
# 用 n 个样本估计(对 c 取期望: 模拟很多组 n-样本)
r = np.random.default_rng(1)
cs = r.binomial(n, p, size=20000)
unbiased = np.mean([pass_at_k_unbiased(n, c, k) for c in cs])
biased = np.mean([pass_at_k_biased(n, c, k) for c in cs])
print(f'真实 pass@{k} (p={p})      = {truth:.4f}')
print(f'无偏估计(组合数)期望      = {unbiased:.4f}  误差 {abs(unbiased-truth):.4f}')
print(f'有偏估计 1-(1-p̂)^k 期望   = {biased:.4f}  误差 {abs(biased-truth):.4f}')
assert abs(unbiased - truth) < 0.005, '组合数估计应无偏(逼近真值)'
assert abs(biased - truth) > abs(unbiased - truth), '有偏版误差更大'
print('✅ 务必用组合数无偏估计 pass@k; 1-(1-p̂)^k 在小样本/难题上有偏(Jensen)')

## 6 · pass@k(覆盖) vs pass^k(可靠) + beam search

pass@k 随 k **升**(覆盖率), pass^k 随 k **降**(全对的可靠性)。两者方向相反、各有用途。
再实现一个用 PRM 剪枝的 **beam search**, 验证它优于同预算的随机采样。

In [ ]:
# pass@k vs pass^k
p = 0.6
print(f"单步正确率 p={p}")
print(f"{'k':>3}{'pass@k(至少1对)':>16}{'pass^k(全对)':>14}")
for k in [1, 2, 4, 8]:
    pak = 1 - (1 - p) ** k        # 至少一个对
    pmk = p ** k                  # 全部对
    print(f'{k:>3}{pak:>16.3f}{pmk:>14.3f}')
assert (1 - (1-p)**8) > (1 - (1-p)**1), 'pass@k 随 k 升'
assert p**8 < p**1, 'pass^k 随 k 降'

# beam search vs 随机采样 (玩具: 逐步构造, PRM=能正确估部分解前景的打分)
from itertools import product
ACTIONS = np.array([0,1,2,3]); K_STEPS=4; TARGET=6; CAP=6; A=4
def is_correct(tr):
    s=0
    for a in tr:
        s+=ACTIONS[a]
        if s>CAP: return 0
    return int(s==TARGET)
def prm_value(prefix):
    '''部分解前景 = 枚举续写答对比例(可枚举所以=真实价值, 模拟一个好 PRM)。'''
    rem=K_STEPS-len(prefix)
    if rem==0: return float(is_correct(prefix))
    if sum(ACTIONS[a] for a in prefix)>CAP: return 0.0
    w=sum(is_correct(list(prefix)+list(c)) for c in product(range(A),repeat=rem))
    return w/(A**rem)

def beam_search(beam_width):
    beams=[[]]
    for _ in range(K_STEPS):
        cand=[b+[a] for b in beams for a in range(A)]
        cand.sort(key=prm_value, reverse=True)
        beams=cand[:beam_width]
    return max(is_correct(b) for b in beams)

def random_search(n_samples, seed):
    r=np.random.default_rng(seed)
    return max(is_correct([int(r.choice(A)) for _ in range(K_STEPS)]) for _ in range(n_samples))

beam_acc = beam_search(beam_width=4)
rand_acc = np.mean([random_search(4, s) for s in range(2000)])
print(f'\nbeam search(宽4, PRM剪枝) 找到正确解 = {beam_acc}')
print(f'随机采样(4条)        找到正确解概率 = {rand_acc:.3f}')
assert beam_acc == 1, '好 PRM 引导的 beam 应稳定找到正确解'
print('✅ pass@k升/pass^k降; beam(PRM剪枝)比同预算随机采样更系统高效')

---
## ✏️ 练习 1：best-of-N

实现 `best_of_n(correct, vscore)`(选 verifier 分最高、返回是否真对) 和 `pass_at_N_oracle(correct)`(上界)。验证 BoN 介于 pass@1 与 pass@N 之间。

In [ ]:
def best_of_n(correct, vscore):
    # TODO: 返回 verifier 分最高那条是否真对
    raise NotImplementedError

def pass_at_N_oracle(correct):
    # TODO: 至少一个对 -> 1
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
correct = np.array([0, 1, 0, 0])
# verifier 给第2条(对的)最高分 -> BoN 选对
assert best_of_n(correct, np.array([0.1, 0.9, 0.2, 0.3])) == 1
# verifier 给错的最高分 -> BoN 选错
assert best_of_n(correct, np.array([0.9, 0.1, 0.2, 0.3])) == 0
assert pass_at_N_oracle(correct) == 1
assert pass_at_N_oracle(np.zeros(5)) == 0
print('✅ 练习 1 通过：best-of-N 与上界 pass@N')

## ✏️ 练习 2：加权多数投票

实现 `weighted_majority(answers, weights)`：按权重给各答案累加、返回总权重最高的答案。对拍：所有权重相等时退化为普通多数投票。

In [ ]:
def weighted_majority(answers, weights):
    # TODO: 累加每个答案的权重, 返回权重最大的答案
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 答案 7 有 2 票但权重低; 答案 3 有 1 票但权重高 -> 加权选 3
assert weighted_majority([7, 7, 3], [0.1, 0.1, 0.9]) == 3
# 等权 -> 退化为多数投票(7 出现 2 次)
assert weighted_majority([7, 7, 3], [1.0, 1.0, 1.0]) == 7
assert weighted_majority([5], [0.5]) == 5
print('✅ 练习 2 通过：加权投票(等权时退化为多数投票)')

## ✏️ 练习 3：pass@k 无偏估计

实现 `pass_at_k_unbiased(n, c, k)` = $1-\binom{n-c}{k}/\binom{n}{k}$(处理 $n-c<k$ 的退化)。验证它逼近真实 pass@k、且优于有偏版。

In [ ]:
from math import comb
def pass_at_k_unbiased(n, c, k):
    # TODO: n-c<k -> 1.0; 否则 1 - C(n-c,k)/C(n,k)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# n=5,c=0,k=3: 没有对的 -> pass@k=0
assert abs(pass_at_k_unbiased(5, 0, 3) - 0.0) < 1e-12
# n=5,c=5,k=3: 全对 -> pass@k=1
assert abs(pass_at_k_unbiased(5, 5, 3) - 1.0) < 1e-12
# n-c<k 退化: n=5,c=4,k=3 -> 错的只有1个<3 -> 必抽到对 -> 1
assert abs(pass_at_k_unbiased(5, 4, 3) - 1.0) < 1e-12
# k=1 -> 就是 c/n
assert abs(pass_at_k_unbiased(10, 3, 1) - 0.3) < 1e-12
# 单调: c 越多 pass@k 越高
assert pass_at_k_unbiased(10, 5, 3) > pass_at_k_unbiased(10, 2, 3)
print('✅ 练习 3 通过：pass@k 无偏估计(评测必备)')

## ✏️ 练习 4：pass^k 可靠性

实现 `pass_pow_k(p, k)` = $p^k$(k 步全对的概率) 和 `steps_for_reliability(p, target_reliability)`(单步正确率 p 下, 要 pass^k≥target 最多能走几步)。这是多步 agent 的可靠性账。

In [ ]:
def pass_pow_k(p, k):
    # TODO: p^k
    raise NotImplementedError

def steps_for_reliability(p, target_reliability):
    # TODO: 最大 k 使 p^k >= target (k>=1); 用对数或循环
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
assert abs(pass_pow_k(0.9, 3) - 0.729) < 1e-9
assert pass_pow_k(0.5, 1) == 0.5
# p=0.9, 要 95% 可靠: 0.9^1=.9<.95? -> 0.9^1=0.9 <0.95 -> 0 步? 取 k>=1 则最大满足的 k
# 0.99^k>=0.95 -> k<=5.1 -> 5
assert steps_for_reliability(0.99, 0.95) == 5
# 单步越可靠, 能走越多步
assert steps_for_reliability(0.999, 0.95) > steps_for_reliability(0.99, 0.95)
print('✅ 练习 4 通过：pass^k 可靠性 —— 多步 agent 的脆弱性账')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def best_of_n(correct, vscore):
    return int(correct[np.argmax(vscore)])

def pass_at_N_oracle(correct):
    return int(np.asarray(correct).sum() > 0)

In [ ]:
# 练习 2 参考答案
def weighted_majority(answers, weights):
    weight = {}
    for a, w in zip(answers, weights):
        weight[a] = weight.get(a, 0.0) + w
    return max(weight, key=weight.get)

In [ ]:
# 练习 3 参考答案
from math import comb
def pass_at_k_unbiased(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - comb(n - c, k) / comb(n, k)

In [ ]:
# 练习 4 参考答案
import math
def pass_pow_k(p, k):
    return p ** k

def steps_for_reliability(p, target_reliability):
    k = 1
    while p ** (k + 1) >= target_reliability:
        k += 1
    return k

---
## 🧪 真实数据胶囊：用真实 pass@k 看『生成易、验证难』

用 Brown 2024《Large Language Monkeys》的真实观察(pass@k 随采样数近似幂律上升)与真实模型的单样本正确率, 算 best-of-N / pass@k 的账。带 try/except 回退到内置真实数值。

In [ ]:
# 真实模型在数学/代码基准上的单样本正确率(公开报告, 约数)
REAL_MODELS = {
    'small-model (GSM8K)': dict(pass1=0.30),
    'mid-model   (GSM8K)': dict(pass1=0.55),
    'strong-model(GSM8K)': dict(pass1=0.80),
}
from math import comb
def pass_at_k_from_p(p, k):
    '''理想: 无穷样本下 pass@k = 1-(1-p)^k (此处 p 是真值不是估计, 故无偏)。'''
    return 1 - (1 - p) ** k

print(f"{'模型':22}{'pass@1':>8}{'pass@8':>8}{'pass@64':>9}")
for name, m in REAL_MODELS.items():
    p = m['pass1']
    print(f'{name:22}{p:>8.2f}{pass_at_k_from_p(p,8):>8.2f}{pass_at_k_from_p(p,64):>9.2f}')
# 关键观察: 即便弱模型, 大量采样后 pass@k(覆盖率)也很高 -> 瓶颈在『挑出对的』
assert pass_at_k_from_p(0.30, 64) > 0.9, '弱模型大量采样后覆盖率也很高'
assert pass_at_k_from_p(0.30, 64) > pass_at_k_from_p(0.30, 1)
print('\n观察: 采样足够多, 连弱模型 pass@k 都接近 1(解在样本里!) -> 瓶颈是 verifier 能否挑出它')

**🧪 胶囊练习**：实现 `bon_gain(p, N)` —— 完美 verifier 下 best-of-N 相对单样本的**正确率增益** = pass@N − pass@1。它量化『多采样能换多少准确率(上界)』。

In [ ]:
def bon_gain(p, N):
    # TODO: (1-(1-p)^N) - p
    raise NotImplementedError

In [ ]:
# 自测
assert abs(bon_gain(0.3, 1) - 0.0) < 1e-12, 'N=1 无增益'
assert bon_gain(0.3, 16) > 0, 'N>1 有增益'
# 中等正确率增益空间最大(极端 p 接近 0/1 时增益小)
assert bon_gain(0.5, 8) > bon_gain(0.95, 8), '已很准的模型 BoN 增益小'
print(f'p=0.3, best-of-16 上界增益 = {bon_gain(0.3,16):.3f}')
print('✅ 胶囊练习通过')

In [ ]:
# 📖 胶囊参考答案
def bon_gain(p, N):
    return (1 - (1 - p) ** N) - p

### 小结
- **搜索 = 生成多候选 + verifier 挑/引导**, 前提『判别比生成易』; 把测试时算力换成准确率。
- **best-of-N**: verifier 选最高分; 上界 pass@N(完美 oracle), 实际看 verifier 多准(完美→pass@N, 全噪声→pass@1)。
- **多数投票**(免 verifier): 对的集中、错的分散 -> 众数选对; **加权投票**兼取鲁棒+判别, 常胜裸 BoN。
- **pass@k** 用**组合数无偏估计** $1-\binom{n-c}{k}/\binom{n}{k}$; $1-(1-\hat p)^k$ 在小样本/难题上**有偏**(Jensen)。
- **pass@k(覆盖,随k升) vs pass^k(可靠,随k降)**: 方向相反、各有用途(BoN 关心前者, 多步 agent 关心后者)。
- **beam/MCTS** 用 PRM 在过程中剪枝, 比独立采样省样本但更依赖 verifier 质量(verifier 越可信越该引导搜索)。

下一站：**模块 05 · 测试时计算 scaling** —— 把『该花多少算力、怎么分配』正式化为 scaling 律与 compute-optimal。